Import Dataset

In [8]:
import pandas as pd

# Construct the raw GitHub URL for the dataset
github_url = "https://raw.githubusercontent.com/ctrazona1385/DS_Capstone_Group_1/main/data/v1_renamed/cdc_prevalence_tractlvl_2022.csv"

# Read the CSV file into a pandas DataFrame
try:
    df_cdc = pd.read_csv(github_url)
    print("Dataset loaded successfully!")
    print("First 5 rows of the dataset:")
    print(df_cdc.head())
    print("\nDataset Info:")
    df_cdc.info()
except Exception as e:
    print(f"Error loading dataset: {e}")


Dataset loaded successfully!
First 5 rows of the dataset:
   index       geoid  uninsured_pct  high_bp_pct  asthma_pct  no_checkup_pct  \
0    226  1001020100           15.0         38.8         9.8            74.1   
1   2109  1001020200           19.9         43.6        10.9            76.5   
2   3955  1001020300           16.7         39.6        10.1            74.3   
3   5732  1001020400           12.5         39.2         8.8            75.5   
4   8015  1001020500           12.8         34.6         9.2            73.5   

   no_dental_visit_pct  diabetes_pct  high_cholesterol_pct  geoid_county  
0                 63.4          10.7                  35.6          1001  
1                 55.6          13.4                  32.7          1001  
2                 61.1          11.3                  35.0          1001  
3                 70.3          10.2                  37.7          1001  
4                 68.4           8.7                  32.7          1001  

Dataset In

Make a copy

In [9]:
df_cdc_copy = df_cdc.copy()
print("DataFrame df_cdc copied to df_cdc_copy.")

DataFrame df_cdc copied to df_cdc_copy.


Generate a data quality report

In [10]:
print("\n--- Data Quality Report ---")
print("\nMissing Values:")
print(df_cdc_copy.isnull().sum())

print("\nData Types:")
print(df_cdc_copy.dtypes)

print("\nDescriptive Statistics:")
print(df_cdc_copy.describe())

print("\nUnique values per column:")
for col in df_cdc_copy.columns:
    print(f"  {col}: {df_cdc_copy[col].nunique()} unique values")


--- Data Quality Report ---

Missing Values:
index                      0
geoid                      0
uninsured_pct              0
high_bp_pct             1999
asthma_pct                 0
no_checkup_pct             0
no_dental_visit_pct        0
diabetes_pct               0
high_cholesterol_pct    1999
geoid_county               0
dtype: int64

Data Types:
index                     int64
geoid                     int64
uninsured_pct           float64
high_bp_pct             float64
asthma_pct              float64
no_checkup_pct          float64
no_dental_visit_pct     float64
diabetes_pct            float64
high_cholesterol_pct    float64
geoid_county              int64
dtype: object

Descriptive Statistics:
              index         geoid  uninsured_pct   high_bp_pct    asthma_pct  \
count  72337.000000  7.233700e+04   72337.000000  70338.000000  72337.000000   
mean   36168.000000  2.782282e+10      15.747539     32.491583     10.031038   
std    20882.037548  1.581816e+10      

Pull all rows where data is not present

Based on data from rows where data is present, logreg or other model the stat

Add imputations to copy of dataset

In [11]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np

# Features to use for imputation
features_for_imputation = [
    'uninsured_pct',
    'asthma_pct',
    'no_checkup_pct',
    'no_dental_visit_pct',
    'diabetes_pct',
    'high_bp_pct',
    'high_cholesterol_pct'
]

# Extract the relevant subset of data
data_to_impute = df_cdc_copy[features_for_imputation].copy()

# 1. Normalize the predictors
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data_to_impute)
scaled_df = pd.DataFrame(scaled_data, columns=features_for_imputation)

# 2. Apply custom weightings
# Weight of 1: uninsured, asthma, no checkup, no dental (default, so multiply by 1 or do nothing)
# Weight of 2: bp, cholesterol, and diabetes
weights = {
    'uninsured_pct': 1,
    'asthma_pct': 1,
    'no_checkup_pct': 1,
    'no_dental_visit_pct': 1,
    'diabetes_pct': 2,
    'high_bp_pct': 2,
    'high_cholesterol_pct': 2
}

# Apply weights by multiplying the scaled columns
for col, weight in weights.items():
    scaled_df[col] = scaled_df[col] * weight

# 3. Perform KNN Imputation
# Using a default k=5, you can adjust this if needed
knn_imputer = KNNImputer(n_neighbors=5)
imputed_scaled_data = knn_imputer.fit_transform(scaled_df)

# 4. Reverse the weighting
imputed_scaled_df = pd.DataFrame(imputed_scaled_data, columns=features_for_imputation)
for col, weight in weights.items():
    imputed_scaled_df[col] = imputed_scaled_df[col] / weight

# 5. Reverse the normalization
imputed_data = scaler.inverse_transform(imputed_scaled_df)
imputed_final_df = pd.DataFrame(imputed_data, columns=features_for_imputation)

# Update the original copy with the imputed values
df_cdc_copy['high_bp_pct'] = imputed_final_df['high_bp_pct']
df_cdc_copy['high_cholesterol_pct'] = imputed_final_df['high_cholesterol_pct'] # Imputes cholesterol too

print("KNN Imputation complete.")
print("\nMissing values after imputation:")
print(df_cdc_copy[features_for_imputation].isnull().sum())

print("\nSummary of imputed 'high_bp_pct':")
print(df_cdc_copy['high_bp_pct'].describe())


KNN Imputation complete.

Missing values after imputation:
uninsured_pct           0
asthma_pct              0
no_checkup_pct          0
no_dental_visit_pct     0
diabetes_pct            0
high_bp_pct             0
high_cholesterol_pct    0
dtype: int64

Summary of imputed 'high_bp_pct':
count    72337.000000
mean        32.451226
std          7.323119
min          5.100000
25%         27.600000
50%         31.800000
75%         36.700000
max         72.300000
Name: high_bp_pct, dtype: float64


In [12]:
print("\nSummary of imputed 'high_cholesterol_pct':")
print(df_cdc_copy['high_cholesterol_pct'].describe())



Summary of imputed 'high_cholesterol_pct':
count    72337.000000
mean        31.815231
std          4.764902
min          6.200000
25%         29.100000
50%         32.100000
75%         34.900000
max         52.900000
Name: high_cholesterol_pct, dtype: float64


Save dataset to CSV

In [13]:
# Save the fully imputed DataFrame to a CSV file
output_filename = 'cdc_prevalence_tractlvl_2022_imputed.csv'
df_cdc_copy.to_csv(output_filename, index=False)

print(f"Dataset successfully saved as '{output_filename}' in the Colab workspace.")

Dataset successfully saved as 'cdc_prevalence_tractlvl_2022_imputed.csv' in the Colab workspace.
